In [ ]:
# experiment from original umap notebook but now with larger data set (MNIST)
# source for code: https://github.com/lmcinnes/umap_paper_notebooks/blob/master/CrossVal%20KNN%20Classifier%20-%20MNIST.ipynb

In [8]:
import numpy as np
import pandas as pd
import sklearn.neighbors
import sklearn.model_selection
import sklearn.datasets
import sklearn.manifold
import sklearn.decomposition

In [11]:
def score_embedding(data, labels, k=10, n_neighbors=5, random_state=42):
    model = sklearn.neighbors.KNeighborsClassifier(n_neighbors=n_neighbors)
    cv = sklearn.model_selection.StratifiedKFold(n_splits=k)
    # only change to original code was removing random_state = random_state, as newer scikit-learn versions require shuffle=True when random_state is set
    # without shuffling, the folds follow the original data order, which reproduces the standard deviations of the paper (same goes for smaller dataset code)
    return sklearn.model_selection.cross_val_score(model, data, labels, cv=cv)

In [12]:
def compare_embeddings(embeddings, n_neighbors=5, k=10, random_state=42):
    cross_val_scores = {}
    for alg in embeddings.keys():
        cross_val_scores[alg] = score_embedding(embeddings[alg][['x', 'y']].values, 
                                                embeddings[alg]['labels'].values,
                                                n_neighbors=n_neighbors,
                                                k=k,
                                                random_state=random_state)
    return pd.DataFrame(cross_val_scores).assign(n_neighbors=n_neighbors)

In [13]:
mnist = {}
mnist['tsne'] = pd.read_csv('embedding_mnist_tsne1.csv', index_col=0)
mnist['umap'] = pd.read_csv('embedding_mnist_umap1.csv', index_col=0)
mnist['lvis'] = pd.read_csv('embedding_mnist_largevis1.csv', index_col=0)
mnist['fits'] = pd.read_csv('embedding_mnist_fitsne1.csv', index_col=0)
mnist['leig'] = pd.read_csv('embedding_mnist_eigenmaps1.csv', index_col=0)
mnist['pca'] = pd.read_csv('embedding_mnist_pca.csv', index_col=0)

In [14]:
df = pd.concat(compare_embeddings(mnist, n_neighbors=n, k=20) for n in (100,200,400,800,1600,3200))

In [17]:
df.groupby('n_neighbors').mean()

,tsne,umap,lvis,fits,leig,pca
n_neighbors,,,,,,
100,0.966829,0.967086,0.962086,0.962429,0.668200,0.462529
200,0.965543,0.967029,0.961500,0.961329,0.667386,0.467171
400,0.963671,0.966843,0.961371,0.961057,0.664171,0.468271
800,0.962886,0.966371,0.961086,0.956900,0.659929,0.468286
1600,0.959143,0.965571,0.947243,0.950086,0.651157,0.466443
3200,0.946071,0.964471,0.920014,0.941843,0.639129,0.459429


In [18]:
df.groupby('n_neighbors').std() * 2

,tsne,umap,lvis,fits,leig,pca
n_neighbors,,,,,,
100,0.014666,0.013451,0.014869,0.014738,0.016062,0.023267
200,0.015249,0.013498,0.014822,0.014769,0.016122,0.023881
400,0.015761,0.013463,0.014996,0.015163,0.016165,0.024313
800,0.016209,0.014082,0.015452,0.016170,0.016915,0.022918
1600,0.016524,0.014359,0.015310,0.017831,0.014787,0.023525
3200,0.017148,0.014517,0.017526,0.018581,0.017354,0.022674


In [ ]:
import seaborn as sns
sns.set()
tidy_df = df.reset_index().melt(id_vars=['index', 'n_neighbors'], 
                                value_vars=['fits', 'leig', 'lvis', 'pca', 'tsne', 'umap'])
tidy_df = tidy_df.rename(columns={'index':'fold', 'variable':'algorithm'})
tidy_df = tidy_df[tidy_df.algorithm != 'fits']

In [ ]:
sns.catplot(x='algorithm', 
            y='value', 
            hue='algorithm', 
            kind='swarm', 
            col='n_neighbors',
            order=['pca', 'leig', 'lvis', 'tsne', 'umap'],
            data=tidy_df)

In [ ]:
tidy_df.assign(dataset='MNIST').to_csv('mnist_crossval_results.csv')